In [ ]:
# ============================================================
# CELL 1: INSTALL PACKAGES
# ============================================================
print("📦 Installing packages...")
!pip install -q --upgrade pip
!pip install -q langchain langchain-community langchain-core langchain-text-splitters
!pip install -q faiss-cpu sentence-transformers transformers accelerate
!pip install -q gradio PyPDF2 pdfplumber
# torch is pre-installed in Colab, no need to reinstall

print("\n✅ All packages installed! Restart runtime if this is first install.")

📦 Installing packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 65.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.

✅ All packages installed! Restart runtime if this is first install.


In [ ]:
# ============================================================
# CELL 2: IMPORT LIBRARIES
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter  # ✅ Fixed
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document  # ✅ Fixed
import gradio as gr
import torch
import PyPDF2
import pdfplumber
import os
import shutil
import pickle

print("✅ All libraries imported successfully!")
print(f"🔥 CUDA available: {torch.cuda.is_available()}")
print(f"💻 Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

✅ All libraries imported successfully!
🔥 CUDA available: True
💻 Device: GPU


In [ ]:
# ============================================================
# CELL 3: PDF EXTRACTION + INITIAL KNOWLEDGE BASE
# ============================================================

def extract_text_from_pdf(pdf_path):
    """Extract text from PDF using PyPDF2 with pdfplumber fallback"""
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            for page in pdf_reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        if text.strip():
            print(f"✅ Extracted {len(text)} characters using PyPDF2")
            return text
    except Exception as e:
        print(f"⚠️ PyPDF2 failed: {e}")

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        print(f"✅ Extracted {len(text)} characters using pdfplumber")
        return text
    except Exception as e:
        print(f"❌ pdfplumber also failed: {e}")
        return ""

print("✅ PDF extraction function loaded!")

# --- Sample knowledge base ---
sample_texts = [
    "The paper 'Attention Is All You Need' introduces the Transformer architecture, which replaces recurrence with self-attention mechanisms.",
    "Transformers use multi-head attention to capture dependencies between words regardless of their position in the sequence.",
    "The model employs positional encoding using sine and cosine functions to retain the order of words in a sentence.",
    "Self-attention allows each word to attend to every other word in the input sequence, enabling parallel computation.",
    "The Transformer architecture consists of an encoder and decoder, each with 6 identical layers.",
    "The model achieves 28.4 BLEU on English-to-German translation and 41.8 BLEU on English-to-French translation.",
    "Training was performed on 8 NVIDIA P100 GPUs for approximately 12 hours for the base model.",
    "The Transformer uses scaled dot-product attention: Attention(Q,K,V) = softmax(QK^T/sqrt(d_k))V",
    "The model uses dropout with rate 0.1 and label smoothing with value 0.1 during training.",
    "Multi-head attention allows the model to jointly attend to information from different representation subspaces.",
    "The model was trained on WMT 2014 English-German dataset with about 4.5 million sentence pairs.",
    "The architecture uses residual connections and layer normalization after each sub-layer.",
    "Feed-forward networks in the Transformer use ReLU activation with inner dimensionality of 2048."
]

print("🔧 Creating embeddings model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

docs = [Document(page_content=text) for text in sample_texts]
vectorstore = FAISS.from_documents(docs, embeddings)

custom_pdf_loaded = False
current_pdf_name = "Sample Knowledge Base (Transformer Paper)"

print(f"✅ Vector store created with {len(sample_texts)} documents")

✅ PDF extraction function loaded!
🔧 Creating embeddings model...


/tmp/ipykernel_5466/1514348780.py:53: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Vector store created with 13 documents


In [ ]:
# ============================================================
# CELL 4A: LOAD MODEL FROM GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

model_path = "/content/drive/MyDrive/OpenLLaMA_Models/openllama3b_model"

if not os.path.exists(model_path):
    print(f"❌ Model not found at: {model_path}")
    print("👉 Run Cell 4B instead to download from HuggingFace.")
else:
    print(f"✅ Found model. Loading...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        low_cpu_mem_usage=True,
        device_map="auto",
        trust_remote_code=True
    )

    # ✅ FIXED: Do NOT set generation params here — set them at call time only
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        pad_token_id=tokenizer.pad_token_id
    )

    print("✅ Model loaded from Drive!")
    print(f"💻 Device: {model.device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Found model. Loading...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/237 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Model loaded from Drive!
💻 Device: cuda:0


In [ ]:
# ============================================================
# CELL 4B: DOWNLOAD MODEL FROM HUGGINGFACE
# ============================================================
llama_model_name = "openlm-research/open_llama_3b"

print("📥 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(llama_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("📥 Loading model (10-15 mins first time)...")
model = AutoModelForCausalLM.from_pretrained(
    llama_model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True,
    device_map="auto",
    trust_remote_code=True
)

# ✅ FIXED: No generation params in pipeline constructor
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    pad_token_id=tokenizer.pad_token_id
)

print("✅ Model loaded from HuggingFace!")
print(f"💻 Device: {model.device}")


🤖 LOADING OPEN-LLAMA 3B MODEL
📥 Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/593 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/534k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/6.85G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/237 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/6.85G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

🔧 Creating text generation pipeline...
✅ Open-LLaMA 3B model loaded successfully!



In [ ]:
# ============================================================
# CELL 5: FIXED RAG PIPELINE
# ============================================================

def add_pdf_to_knowledge_base(pdf_file):
    global vectorstore, custom_pdf_loaded, current_pdf_name

    if pdf_file is None:
        return "⚠️ No PDF file uploaded."

    try:
        pdf_path = pdf_file if isinstance(pdf_file, str) else pdf_file.name
        text = extract_text_from_pdf(pdf_path)

        if not text or len(text) < 100:
            return "❌ Could not extract sufficient text. Ensure the PDF is text-based, not scanned."

        print(f"📊 Extracted {len(text)} characters, ~{len(text.split())} words")

        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=600,
            chunk_overlap=150,
            length_function=len
        )
        chunks = text_splitter.split_text(text)
        print(f"📦 Created {len(chunks)} chunks")

        docs = [Document(page_content=chunk) for chunk in chunks]
        vectorstore = FAISS.from_documents(docs, embeddings)

        custom_pdf_loaded = True
        current_pdf_name = os.path.basename(pdf_path)

        return f"""✅ **PDF Successfully Processed!**

📄 **File:** {current_pdf_name}
📊 **Characters:** {len(text):,}
📊 **Words:** {len(text.split()):,}
📦 **Chunks:** {len(chunks)}
🗄️ **Knowledge base:** Updated to use ONLY this PDF"""

    except Exception as e:
        import traceback
        return f"❌ Error: {str(e)}\n{traceback.format_exc()}"


def generate_answer(query, num_context_chunks=4):
    """✅ FIXED: Proper answer extraction and generation params"""
    if not query or len(query.strip()) < 3:
        return "⚠️ Please enter a valid question."

    try:
        print(f"\n🔍 Query: {query}")

        docs = vectorstore.similarity_search(query, k=int(num_context_chunks))
        context = "\n\n".join([d.page_content for d in docs])

        print(f"📚 Retrieved {len(docs)} chunks ({len(context)} chars)")

        # ✅ FIXED: Cleaner prompt that works better with Open-LLaMA
        prompt = f"""### Instruction:
You are a research assistant. Answer the question below using ONLY the provided context.
If the answer is not found in the context, respond with: "This information is not available in the document."

### Context:
{context}

### Question:
{query}

### Answer:
"""

        # ✅ FIXED: All generation params passed at call time, with do_sample=True
        result = pipe(
            prompt,
            max_new_tokens=350,
            do_sample=True,
            temperature=0.2,       # Low = more faithful to context
            top_p=0.85,
            repetition_penalty=1.3,
            return_full_text=False  # ✅ KEY FIX: Returns ONLY the new tokens, not the prompt
        )

        # ✅ FIXED: With return_full_text=False, result is clean
        answer = result[0]['generated_text'].strip()

        # Fallback if empty
        if not answer or len(answer) < 15:
            answer = "Could not generate a clear answer. Try rephrasing your question or increasing context chunks."

        # Append source info
        source_info = f"\n\n---\n**📄 Source:** {current_pdf_name}"
        source_info += f"\n**📦 Chunks used:** {len(docs)}"
        source_info += f"\n\n**📝 Retrieved context preview:**\n_{context[:300]}..._"

        print("✅ Answer generated!")
        return answer + source_info

    except Exception as e:
        import traceback
        return f"❌ Error: {str(e)}\n{traceback.format_exc()}"


def reset_to_sample_knowledge():
    global vectorstore, custom_pdf_loaded, current_pdf_name
    docs = [Document(page_content=text) for text in sample_texts]
    vectorstore = FAISS.from_documents(docs, embeddings)
    custom_pdf_loaded = False
    current_pdf_name = "Sample Knowledge Base (Transformer Paper)"
    return "✅ Reset to sample Transformer paper knowledge base"

print("✅ RAG pipeline loaded!")

✅ RAG pipeline loaded!


In [ ]:
# ============================================================
# CELL 6: GRADIO INTERFACE
# ============================================================

with gr.Blocks(theme=gr.themes.Soft(), title="Research Paper Q&A") as demo:

    gr.Markdown("# 🔬 Research Paper Q&A Assistant\n**Open-LLaMA 3B + RAG**")
    kb_display = gr.Markdown(f"**Currently using:** {current_pdf_name}")

    with gr.Tabs():
        with gr.Tab("💬 Ask Questions"):
            with gr.Row():
                with gr.Column(scale=1):
                    query_input = gr.Textbox(
                        label="❓ Your Question",
                        placeholder="e.g., What is the main contribution of this paper?",
                        lines=4
                    )
                    # ✅ FIXED: Define slider outside nested scope for reliable reference
                    num_chunks = gr.Slider(minimum=2, maximum=6, value=4, step=1,
                                          label="📚 Context Chunks (recommended: 4)")
                    submit_btn = gr.Button("🚀 Get Answer", variant="primary")

                with gr.Column(scale=2):
                    answer_output = gr.Textbox(label="💡 Answer", lines=18, interactive=False)

            gr.Examples(
                examples=[
                    ["What is the main contribution of this paper?"],
                    ["What methodology or approach is used?"],
                    ["What are the key results or findings?"],
                    ["What datasets were used in this research?"],
                    ["What are the limitations mentioned?"],
                ],
                inputs=query_input
            )

        with gr.Tab("📄 Upload PDF"):
            gr.Markdown("### Upload your research paper PDF\n⚠️ Must be text-based PDF, not scanned image.")
            pdf_input = gr.File(label="📎 Upload PDF", file_types=[".pdf"], type="filepath")
            with gr.Row():
                upload_btn = gr.Button("📥 Process PDF", variant="primary")
                reset_btn = gr.Button("🔄 Reset to Sample", variant="secondary")
            upload_output = gr.Markdown("Ready to process PDF...")

            upload_btn.click(fn=add_pdf_to_knowledge_base, inputs=pdf_input, outputs=upload_output)
            reset_btn.click(fn=reset_to_sample_knowledge, inputs=None, outputs=upload_output)

        with gr.Tab("ℹ️ System Info"):
            gr.Markdown(f"""
## System Configuration
- **Model**: Open-LLaMA 3B
- **Embeddings**: all-MiniLM-L6-v2
- **Vector DB**: FAISS
- **Device**: {'GPU ✅' if torch.cuda.is_available() else 'CPU ⚠️'}
- **Chunk size**: 600 chars, 150 overlap
- **Temperature**: 0.2 (context-faithful)
- **`return_full_text`**: False (clean output)

## Tips
- Use text-based PDFs only (not scanned)
- Increase chunks for longer answers
- Reset before uploading a new PDF
            """)

    # ✅ FIXED: Click connected at top-level scope, not inside Tab
    submit_btn.click(
        fn=generate_answer,
        inputs=[query_input, num_chunks],
        outputs=answer_output
    )

demo.launch(share=True, debug=True, show_error=True)

/tmp/ipykernel_5466/744188439.py:5: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Research Paper Q&A") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://766a9cb8a39f1d8a4d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ Extracted 39487 characters using PyPDF2
📊 Extracted 39487 characters, ~5998 words
📦 Created 89 chunks


Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'top_p', 'max_new_tokens', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



🔍 Query: What is the main contribution of this paper?
📚 Retrieved 4 chunks (2235 chars)


...


🔍 Query: Explain the main architecture.
📚 Retrieved 4 chunks (2231 chars)
✅ Answer generated!
✅ Extracted 20495 characters using PyPDF2
📊 Extracted 20495 characters, ~2718 words
📦 Created 47 chunks


Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔍 Query: What is the main contribution of this paper?
📚 Retrieved 4 chunks (2033 chars)
✅ Answer generated!


Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔍 Query: What is the name of the interview assistant?
📚 Retrieved 4 chunks (2252 chars)
✅ Answer generated!
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://766a9cb8a39f1d8a4d.gradio.live


In [ ]:
# ============================================================
# CELL 7: SAVE MODEL TO GOOGLE DRIVE (run once after 4B)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

save_path = "/content/openllama3b_model"
os.makedirs(save_path, exist_ok=True)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Saved locally to {save_path}")

drive_path = "/content/drive/MyDrive/OpenLLaMA_Models/openllama3b_model"
if os.path.exists(drive_path):
    shutil.rmtree(drive_path)
shutil.copytree(save_path, drive_path)
print(f"✅ Copied to Google Drive: {drive_path}")